# 01 — Coleta de Dados

Fontes:
- **BCB/SGS** — câmbio USD/BRL, Selic e IPCA via API REST
- **CEPEA/ESALQ** — preços mensais de soja e milho (Paraná)

O CEPEA não tem API. Download manual em https://www.cepea.esalq.usp.br/br/consultas-ao-banco-de-dados-do-site.aspx  
Salvar os arquivos como `data/raw/cepea_soja.xlsx` e `data/raw/cepea_milho.xlsx` antes de rodar este notebook.

In [8]:
import time
import logging
from pathlib import Path
from datetime import date, timedelta

import requests
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger(__name__)

RAW_DIR = Path('../data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

---
## 1. BCB — Séries macro via API SGS

In [9]:
# 1     → câmbio USD/BRL (diário)
# 21619 → Selic Over % a.a.
# 433   → IPCA % ao mês
BCB_SERIES = {'usd_brl': 1, 'selic': 21619, 'ipca': 433}
BCB_URL = 'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{serie}/dados'


def fetch_bcb(serie_code, start_date, end_date, max_retries=3, backoff=2.0):
    url = BCB_URL.format(serie=serie_code)
    params = {'formato': 'json', 'dataInicial': start_date, 'dataFinal': end_date}
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, params=params, timeout=30)
            resp.raise_for_status()
            data = resp.json()
            if not data:
                return pd.DataFrame(columns=['data', 'valor'])
            df = pd.DataFrame(data)
            df['data']  = pd.to_datetime(df['data'], dayfirst=True)
            df['valor'] = pd.to_numeric(df['valor'], errors='coerce')
            return df.sort_values('data').reset_index(drop=True)
        except requests.exceptions.RequestException as exc:
            log.error('Tentativa %d/%d falhou: %s', attempt, max_retries, exc)
            if attempt < max_retries:
                time.sleep(backoff ** attempt)
    return pd.DataFrame(columns=['data', 'valor'])

In [10]:
END_DATE   = date.today()
START_DATE = END_DATE - timedelta(days=5 * 365)
start_str  = START_DATE.strftime('%d/%m/%Y')
end_str    = END_DATE.strftime('%d/%m/%Y')

bcb_dfs = {}
for nome, codigo in BCB_SERIES.items():
    df = fetch_bcb(codigo, start_str, end_str)
    if not df.empty:
        df = df.rename(columns={'valor': nome})
        bcb_dfs[nome] = df
        df.to_parquet(RAW_DIR / f'bcb_{nome}.parquet', index=False)
        log.info('%s: %d registros', nome, len(df))

df_bcb = list(bcb_dfs.values())[0][['data']].copy()
for nome, df in bcb_dfs.items():
    df_bcb = df_bcb.merge(df[['data', nome]], on='data', how='outer')
df_bcb = df_bcb.sort_values('data').reset_index(drop=True)
df_bcb.to_parquet(RAW_DIR / 'bcb_macro.parquet', index=False)

print(f'bcb_macro: {df_bcb.shape}')
df_bcb.tail(3)

16:20:26 [INFO] usd_brl: 1255 registros
16:20:27 [INFO] selic: 1255 registros
16:20:27 [INFO] ipca: 60 registros


bcb_macro: (1281, 4)


,data,usd_brl,selic,ipca
1278,2026-06-24,5.2098,5.9126,NaN
1279,2026-06-25,5.1892,5.9079,NaN
1280,2026-06-26,5.1695,5.8953,NaN


---
## 2. CEPEA — Preços de soja e milho

Dados baixados manualmente no portal CEPEA (periodicidade mensal, Paraná).  
Layout do arquivo: cabeçalho na linha 3 (0-indexed), data no formato `MM/YYYY`, preços como string com vírgula decimal.

In [11]:
def parse_cepea(path, commodity):
    if not path.exists():
        log.error('Arquivo não encontrado: %s', path)
        return pd.DataFrame()

    # Cabeçalho na linha 3, colunas fixas: Data | À vista R$ | À vista US$
    df = pd.read_excel(path, header=3)
    df.columns = ['data', 'preco_rs', 'preco_usd']
    df = df.dropna(subset=['data'])

    # Data no formato MM/YYYY
    df['data'] = pd.to_datetime(df['data'], format='%m/%Y', errors='coerce')
    df = df.dropna(subset=['data'])

    # Preços vêm como string com vírgula decimal — ex: '78,46'
    for col in ['preco_rs', 'preco_usd']:
        df[col] = (
            df[col].astype(str)
            .str.replace('.', '', regex=False)
            .str.replace(',', '.', regex=False)
        )
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df['commodity'] = commodity
    df = df.sort_values('data').reset_index(drop=True)
    log.info('CEPEA %s: %d registros | %s -> %s', commodity, len(df), df['data'].min().date(), df['data'].max().date())
    return df


cepea_dfs = {}
for commodity in ['soja', 'milho']:
    df = parse_cepea(RAW_DIR / f'cepea_{commodity}.xlsx', commodity)
    if not df.empty:
        cepea_dfs[commodity] = df
        df.to_parquet(RAW_DIR / f'cepea_{commodity}.parquet', index=False)

for commodity, df in cepea_dfs.items():
    display(df.head(3))

16:20:33 [INFO] CEPEA soja: 126 registros | 2016-01-01 -> 2026-06-01
16:20:33 [INFO] CEPEA milho: 126 registros | 2016-01-01 -> 2026-06-01


,data,preco_rs,preco_usd,commodity
0,2016-01-01,78.46,19.36,soja
1,2016-02-01,73.32,18.46,soja
2,2016-03-01,69.95,18.93,soja


,data,preco_rs,preco_usd,commodity
0,2016-01-01,41.65,10.27,milho
1,2016-02-01,42.98,10.82,milho
2,2016-03-01,47.79,12.94,milho


---
## 3. Consolidação: preço + câmbio

In [12]:
# Reamosta câmbio diário para mensal e normaliza para primeiro dia do mês
df_cambio = (
    bcb_dfs['usd_brl']
    .set_index('data')
    .resample('ME')
    .last()
    .reset_index()
)
df_cambio['data'] = df_cambio['data'].dt.to_period('M').dt.to_timestamp()

for commodity, df_cepea in cepea_dfs.items():
    df_merged = df_cepea.merge(df_cambio, on='data', how='left')
    df_merged['preco_usd_calc'] = df_merged['preco_rs'] / df_merged['usd_brl']
    df_merged.to_parquet(RAW_DIR / f'cepea_{commodity}_cambio.parquet', index=False)
    print(f'{commodity}: {len(df_merged)} registros | nulos preco_rs: {df_merged["preco_rs"].isna().sum()} | nulos cambio: {df_merged["usd_brl"].isna().sum()}')
    display(df_merged.tail(3))

print('\ndata/raw/:')
for f in sorted(RAW_DIR.glob('*.parquet')):
    print(f'  {f.name:<45} {len(pd.read_parquet(f)):>6} registros')

print('\nColeta concluida.')

soja: 126 registros | nulos preco_rs: 0 | nulos cambio: 65


,data,preco_rs,preco_usd,commodity,usd_brl,preco_usd_calc
123,2026-04-01,121.35,24.12,soja,4.9886,24.325462
124,2026-05-01,123.03,24.68,soja,5.0569,24.329134
125,2026-06-01,124.56,24.34,soja,5.1695,24.095174


milho: 126 registros | nulos preco_rs: 0 | nulos cambio: 65


,data,preco_rs,preco_usd,commodity,usd_brl,preco_usd_calc
123,2026-04-01,67.95,13.50,milho,4.9886,13.621056
124,2026-05-01,65.60,13.16,milho,5.0569,12.972374
125,2026-06-01,63.78,12.46,milho,5.1695,12.337750



data/raw/:
  bcb_ipca.parquet                                  60 registros
  bcb_macro.parquet                               1281 registros
  bcb_selic.parquet                               1255 registros
  bcb_usd_brl.parquet                             1255 registros
  cepea_milho.parquet                              126 registros
  cepea_milho_cambio.parquet                       126 registros
  cepea_soja.parquet                               126 registros
  cepea_soja_cambio.parquet                        126 registros

Coleta concluida.
